In [2]:
# ============================================================
# SECTION 1: LOAD AND PREPARE DATA
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load dataset
df = pd.read_csv("../data/fraudTrain.csv")

print("Dataset shape:", df.shape)

# 2. Convert dates
df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)
df["dob"] = pd.to_datetime(df["dob"])

# 3. Feature engineering

# Transaction hour
df["hour"] = df["trans_date_trans_time"].dt.hour

# Day of week
df["day_of_week"] = df["trans_date_trans_time"].dt.day_name()

# Customer age at transaction
df["age"] = (
    (df["trans_date_trans_time"] - df["dob"]).dt.days
    / 365.25
).astype(int)

# Distance between customer and merchant (Haversine)
R = 6371  # Earth's radius in kilometers

lat1 = np.radians(df["lat"])
lon1 = np.radians(df["long"])
lat2 = np.radians(df["merch_lat"])
lon2 = np.radians(df["merch_long"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

df["distance_km"] = 2 * R * np.arcsin(
    np.sqrt(np.clip(a, 0, 1))
)

# 4. Select features and target
features = [
    "amt",
    "category",
    "hour",
    "day_of_week",
    "age",
    "distance_km"
]

X = df[features]
y = df["is_fraud"]

# 5. Same train/test split as previous notebook
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 6. Define preprocessing
numeric_features = [
    "amt",
    "hour",
    "age",
    "distance_km"
]

categorical_features = [
    "category",
    "day_of_week"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

# 7. Create validation split from training data
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# 8. Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# 9. Verify results
print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nFraud distribution:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())

print("\nProcessed feature shape:", X_train_processed.shape)

Dataset shape: (1296675, 23)

Training samples: 829872
Validation samples: 207468
Testing samples: 259335

Fraud distribution:
Train: 0.00578884454470087
Validation: 0.00578884454470087
Test: 0.005787880540613492

Processed feature shape: (829872, 25)


In [3]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 12.3 MB/s eta 0:00:04
   --- ------------------------------------ 4.7/48.9 MB 18.1 MB/s eta 0:00:03
   ------- -------------------------------- 8.9/48.9 MB 18.6 MB/s eta 0:00:03
   ------------ --------------------------- 15.2/48.9 MB 22.2 MB/s eta 0:00:02
   ----------------- ---------------------- 21.8/48.9 MB 24.6 MB/s eta 0:00:02
   ----------------------- ---------------- 28.6/48.9 MB 26.2 MB/s eta 0:00:01
   ---------------------------- ----------- 35.4/48.9 MB 27.5 MB/s eta 0:00:01
   ---------------------------------- ----- 41.9/48.9 MB 28.2 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 26.9 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42
)

xgb_model.fit(X_train_processed, y_train)

print("XGBoost training complete!")

XGBoost training complete!


In [6]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Predict whether each transaction is fraud (0 or 1)
y_pred = xgb_model.predict(X_val_processed)

# Predict the probability of fraud
y_prob = xgb_model.predict_proba(X_val_processed)[:, 1]

# Evaluate performance
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_pred, digits=4))

print("PR-AUC:", average_precision_score(y_val, y_prob))

Confusion Matrix:
[[206178     89]
 [   240    961]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9988    0.9996    0.9992    206267
           1     0.9152    0.8002    0.8538      1201

    accuracy                         0.9984    207468
   macro avg     0.9570    0.8999    0.9265    207468
weighted avg     0.9984    0.9984    0.9984    207468

PR-AUC: 0.9047249697263556


In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    predictions = (y_prob >= threshold).astype(int)

    precision = precision_score(y_val, predictions)
    recall = recall_score(y_val, predictions)
    f1 = f1_score(y_val, predictions)

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold: 0.2 | Precision: 0.8139 | Recall: 0.8485 | F1: 0.8308
Threshold: 0.3 | Precision: 0.8747 | Recall: 0.8310 | F1: 0.8523
Threshold: 0.4 | Precision: 0.8985 | Recall: 0.8110 | F1: 0.8525
Threshold: 0.5 | Precision: 0.9152 | Recall: 0.8002 | F1: 0.8538
Threshold: 0.6 | Precision: 0.9331 | Recall: 0.7777 | F1: 0.8483


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

for depth in [4, 6, 8]:
    model = XGBClassifier(
        n_estimators=200,
        max_depth=depth,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42
    )

    # Train model
    model.fit(X_train_processed, y_train)

    # Predict on validation data
    y_prob_depth = model.predict_proba(X_val_processed)[:, 1]
    y_pred_depth = (y_prob_depth >= 0.5).astype(int)

    # Evaluate
    print(f"\nMax Depth: {depth}")
    print(f"Precision: {precision_score(y_val, y_pred_depth):.4f}")
    print(f"Recall: {recall_score(y_val, y_pred_depth):.4f}")
    print(f"F1: {f1_score(y_val, y_pred_depth):.4f}")
    print(f"PR-AUC: {average_precision_score(y_val, y_prob_depth):.4f}")


Max Depth: 4
Precision: 0.9090
Recall: 0.7818
F1: 0.8406
PR-AUC: 0.9022

Max Depth: 6
Precision: 0.9152
Recall: 0.8002
F1: 0.8538
PR-AUC: 0.9047

Max Depth: 8
Precision: 0.9114
Recall: 0.7968
F1: 0.8503
PR-AUC: 0.9066


In [12]:
for rate in [0.05, 0.1, 0.2]:
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=rate,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42
    )

    model.fit(X_train_processed, y_train)

    y_prob_rate = model.predict_proba(X_val_processed)[:, 1]
    y_pred_rate = (y_prob_rate >= 0.5).astype(int)

    print(f"\nLearning Rate: {rate}")
    print(f"Precision: {precision_score(y_val, y_pred_rate):.4f}")
    print(f"Recall: {recall_score(y_val, y_pred_rate):.4f}")
    print(f"F1: {f1_score(y_val, y_pred_rate):.4f}")
    print(f"PR-AUC: {average_precision_score(y_val, y_prob_rate):.4f}")


Learning Rate: 0.05
Precision: 0.9081
Recall: 0.7985
F1: 0.8498
PR-AUC: 0.9063

Learning Rate: 0.1
Precision: 0.9152
Recall: 0.8002
F1: 0.8538
PR-AUC: 0.9047

Learning Rate: 0.2
Precision: 0.9054
Recall: 0.7893
F1: 0.8434
PR-AUC: 0.8857


In [13]:
configurations = [
    (0.1, 200),
    (0.05, 300),
    (0.05, 400)
]

for rate, trees in configurations:
    model = XGBClassifier(
        n_estimators=trees,
        max_depth=6,
        learning_rate=rate,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42
    )

    model.fit(X_train_processed, y_train)

    y_prob_tuned = model.predict_proba(X_val_processed)[:, 1]
    y_pred_tuned = (y_prob_tuned >= 0.5).astype(int)

    print(f"\nLearning Rate: {rate}, Trees: {trees}")
    print(f"Precision: {precision_score(y_val, y_pred_tuned):.4f}")
    print(f"Recall: {recall_score(y_val, y_pred_tuned):.4f}")
    print(f"F1: {f1_score(y_val, y_pred_tuned):.4f}")
    print(f"PR-AUC: {average_precision_score(y_val, y_prob_tuned):.4f}")


Learning Rate: 0.1, Trees: 200
Precision: 0.9152
Recall: 0.8002
F1: 0.8538
PR-AUC: 0.9047

Learning Rate: 0.05, Trees: 300
Precision: 0.9076
Recall: 0.8018
F1: 0.8515
PR-AUC: 0.9085

Learning Rate: 0.05, Trees: 400
Precision: 0.9081
Recall: 0.8060
F1: 0.8540
PR-AUC: 0.9097


In [14]:
best_xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42
)

# Train our best candidate
best_xgb.fit(X_train_processed, y_train)

# Get fraud probabilities
best_y_prob = best_xgb.predict_proba(X_val_processed)[:, 1]

# Compare thresholds
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    predictions = (best_y_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision_score(y_val, predictions):.4f}")
    print(f"Recall: {recall_score(y_val, predictions):.4f}")
    print(f"F1: {f1_score(y_val, predictions):.4f}")


Threshold: 0.2
Precision: 0.8154
Recall: 0.8535
F1: 0.8340

Threshold: 0.3
Precision: 0.8678
Recall: 0.8360
F1: 0.8516

Threshold: 0.4
Precision: 0.8936
Recall: 0.8185
F1: 0.8544

Threshold: 0.5
Precision: 0.9081
Recall: 0.8060
F1: 0.8540

Threshold: 0.6
Precision: 0.9277
Recall: 0.7802
F1: 0.8476


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

rf_comparison = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_comparison.fit(X_train_processed, y_train)

rf_prob = rf_comparison.predict_proba(X_val_processed)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)

print("Random Forest Validation Results:")
print(f"Precision: {precision_score(y_val, rf_pred):.4f}")
print(f"Recall: {recall_score(y_val, rf_pred):.4f}")
print(f"F1: {f1_score(y_val, rf_pred):.4f}")
print(f"PR-AUC: {average_precision_score(y_val, rf_prob):.4f}")

Random Forest Validation Results:
Precision: 0.9345
Recall: 0.7727
F1: 0.8459
PR-AUC: 0.8949


In [16]:
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    predictions = (rf_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision_score(y_val, predictions):.4f}")
    print(f"Recall: {recall_score(y_val, predictions):.4f}")
    print(f"F1: {f1_score(y_val, predictions):.4f}")


Threshold: 0.2
Precision: 0.7918
Recall: 0.8518
F1: 0.8207

Threshold: 0.3
Precision: 0.8571
Recall: 0.8243
F1: 0.8404

Threshold: 0.4
Precision: 0.8965
Recall: 0.8002
F1: 0.8456

Threshold: 0.5
Precision: 0.9345
Recall: 0.7727
F1: 0.8459

Threshold: 0.6
Precision: 0.9582
Recall: 0.7452
F1: 0.8384


In [17]:
# Combine training and validation datasets
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

# Refit preprocessing on the full training dataset
X_train_full_processed = preprocessor.fit_transform(X_train_full)

# Create final XGBoost model
final_xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42
)

# Train on full 80%
final_xgb.fit(X_train_full_processed, y_train_full)

print("Final XGBoost model trained!")

Final XGBoost model trained!


In [18]:
# Process test data using the final fitted preprocessor
X_test_final = preprocessor.transform(X_test)

# Predict fraud probabilities
test_prob = final_xgb.predict_proba(X_test_final)[:, 1]

# Apply our selected threshold
test_pred = (test_prob >= 0.4).astype(int)

# Evaluate
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

print("PR-AUC:", average_precision_score(y_test, test_prob))

Confusion Matrix:
[[257705    129]
 [   277   1224]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9989    0.9995    0.9992    257834
           1     0.9047    0.8155    0.8577      1501

    accuracy                         0.9984    259335
   macro avg     0.9518    0.9075    0.9285    259335
weighted avg     0.9984    0.9984    0.9984    259335

PR-AUC: 0.9154337650555925


In [19]:
import joblib
from pathlib import Path

# Create models directory if needed
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save model, preprocessing, and threshold
model_bundle = {
    "model": final_xgb,
    "preprocessor": preprocessor,
    "threshold": 0.4,
    "features": features
}

joblib.dump(
    model_bundle,
    models_dir / "xgboost_final.pkl"
)

print("Final XGBoost model saved successfully!")

Final XGBoost model saved successfully!


In [20]:
import joblib

# Load saved model bundle
saved_bundle = joblib.load("../models/xgboost_final.pkl")

loaded_model = saved_bundle["model"]
loaded_preprocessor = saved_bundle["preprocessor"]
loaded_threshold = saved_bundle["threshold"]

# Transform test data
X_test_loaded = loaded_preprocessor.transform(X_test)

# Make predictions
loaded_prob = loaded_model.predict_proba(X_test_loaded)[:, 1]
loaded_pred = (loaded_prob >= loaded_threshold).astype(int)

# Verify predictions match the original model
print("Predictions match:", np.array_equal(test_pred, loaded_pred))
print("Saved threshold:", loaded_threshold)

Predictions match: True
Saved threshold: 0.4
